# Investigating PokeEvents: Is There a Real, Separate Poke-Out Event? (All 5 Rats)

**Why this matters, urgently:** everything built so far (every window, every offset, every "t=0")
assumes a single reference point per trial, what we've called "Poke-In," taken from the `InSeqLog`
marker and confirmed (way back in notebook 01) to coincide with odor onset. Recent lab feedback raised
the possibility of a distinct "Poke-Out" event, the moment the rat's nose leaves the port, ending that
sniff, and suggested some analyses should be centered on Poke-Out instead of Poke-In.

**The clue we already had, but never followed up on:** `PokeEvents` fires exactly TWICE per trial (584
events for Mitt's 292 trials, confirmed back in the original data audit), We guessed at the time this
meant "one poke to initiate, one poke to respond," but never actually separated the two events or
measured the gap between them. This notebook does that directly, across all 5 rats.

**This notebook only needs each rat's small `bvr` file** (no LFP loading), so it's fast, just seconds
to run, appropriate for a quick, foundational check before any bigger redesign.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
from pathlib import Path

from src.preprocessing import build_labels, get_sampling_rate

raw_dir = Path('../data/raw')
session_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()])
print(f"Found {len(session_dirs)} sessions")


## 1. For Each Rat: Find PokeEvents Pairs Around Each Trial

For every trial, we already know the Poke-In moment (`InSeqLog`'s trial marker, `trial_idx`). We look at
`PokeEvents` in a small window around and after that marker, expecting to find two events, the first
matching Poke-In (as a sanity check), the second being the candidate Poke-Out.


In [ ]:
all_results = {}

for session_dir in session_dirs:
    session_name = session_dir.name
    bvr = np.load(session_dir / f'{session_name}_bvr.npz', allow_pickle=True)
    bvr_data = bvr['data']
    bvr_keys = bvr['keys'].tolist()
    timebin = bvr_data[bvr_keys.index('TimeBin')]
    labels = build_labels(bvr_data, bvr_keys)
    trial_idx = labels['trial_idx']

    poke_events_row = bvr_data[bvr_keys.index('PokeEvents')]
    poke_event_idx = np.where(poke_events_row != 0)[0]

    poke_in_offsets_ms = []   # gap between InSeqLog marker and the nearest PokeEvents entry (sanity check)
    poke_out_gaps_ms = []     # gap between that first PokeEvents entry and the SECOND one (candidate duration)
    sign_pairs = []
    unmatched_trials = 0

    for t in trial_idx:
        # events at or shortly after the InSeqLog marker (allow a small lookback in case of tiny jitter)
        candidates = poke_event_idx[poke_event_idx >= t - 5]
        if len(candidates) < 2:
            unmatched_trials += 1
            continue
        first_idx = candidates[0]
        second_idx = candidates[1]

        poke_in_offsets_ms.append((timebin[first_idx] - timebin[t]) * 1000)
        poke_out_gaps_ms.append((timebin[second_idx] - timebin[first_idx]) * 1000)
        sign_pairs.append((poke_events_row[first_idx], poke_events_row[second_idx]))

    poke_in_offsets_ms = np.array(poke_in_offsets_ms)
    poke_out_gaps_ms = np.array(poke_out_gaps_ms)

    all_results[session_name] = {
        'n_trials': len(trial_idx),
        'n_matched_pairs': len(poke_out_gaps_ms),
        'n_unmatched': unmatched_trials,
        'poke_in_offset_median_ms': float(np.median(poke_in_offsets_ms)) if len(poke_in_offsets_ms) else None,
        'poke_in_offset_max_abs_ms': float(np.max(np.abs(poke_in_offsets_ms))) if len(poke_in_offsets_ms) else None,
        'gap_median_ms': float(np.median(poke_out_gaps_ms)) if len(poke_out_gaps_ms) else None,
        'gap_min_ms': float(np.min(poke_out_gaps_ms)) if len(poke_out_gaps_ms) else None,
        'gap_max_ms': float(np.max(poke_out_gaps_ms)) if len(poke_out_gaps_ms) else None,
        'gap_std_ms': float(np.std(poke_out_gaps_ms)) if len(poke_out_gaps_ms) else None,
        'sign_pairs_seen': list(set(tuple(float(x) for x in p) for p in sign_pairs)),
    }

    print(f"\n{session_name}")
    print(f"  Trials: {len(trial_idx)}, matched pairs: {len(poke_out_gaps_ms)}, unmatched: {unmatched_trials}")
    if len(poke_in_offsets_ms):
        print(f"  First PokeEvents entry vs InSeqLog marker: median offset {np.median(poke_in_offsets_ms):.2f}ms "
              f"(max abs {np.max(np.abs(poke_in_offsets_ms)):.2f}ms) -- should be ~0 if these are the same event")
    if len(poke_out_gaps_ms):
        print(f"  Gap between 1st and 2nd PokeEvents entry: median {np.median(poke_out_gaps_ms):.1f}ms "
              f"(min {np.min(poke_out_gaps_ms):.1f}, max {np.max(poke_out_gaps_ms):.1f}, std {np.std(poke_out_gaps_ms):.1f})")
    print(f"  Sign values seen (1st entry, 2nd entry): {all_results[session_name]['sign_pairs_seen']}")


## 2. What to Look For in the Output Above

- **"First PokeEvents entry vs InSeqLog marker" should be very close to 0ms.** If it is, that confirms
  the first `PokeEvents` entry is the SAME moment as what we've been calling Poke-In all along, good, our
  existing t=0 reference is correctly anchored to nose-in.
- **The gap between the 1st and 2nd entry is the candidate Poke-In-to-Poke-Out duration**, i.e., how long
  the rat's nose stayed in the port. If this is small and consistent (say, a few hundred ms, low
  variability), that's a real, usable, distinct second reference point. If it's huge or wildly
  inconsistent, the "2nd PokeEvents entry" may not actually be Poke-Out, it could be something else
  entirely (e.g. a response-port poke elsewhere, not related to sniff duration).
- **Sign values**: if the first and second entries have consistently different signs (e.g. always +1
  then -1, or vice versa), that's strong independent evidence they represent two different physical
  events (nose in vs. nose out), not two readings of the same event.


## 3. Text-Only Results Export


In [ ]:
import json as _json
import os

print(_json.dumps(all_results, indent=2))

os.makedirs('../outputs/logs', exist_ok=True)
with open('../outputs/logs/notebook019_pokeout_investigation.json', 'w') as f:
    _json.dump(all_results, f, indent=2)
print("\nSaved to outputs/logs/notebook019_pokeout_investigation.json")


## 4. Written Summary (Fill In After Running)

**Question:** does a real, distinct Poke-Out event exist in this data, separate from Poke-In?

**Evidence found:** *(fill in from Section 1's output)*
- Does the first PokeEvents entry line up with InSeqLog (confirming our current Poke-In reference is
  correct)?
- Is the gap to the second entry small and consistent (supporting "sniff duration") or large/inconsistent
  (suggesting it's something else)?
- Do the sign values distinguish the two events?

**Conclusion and next step:** *(fill in)* If Poke-Out is confirmed as a real, distinct, consistently-
timed event, the next notebook should test whether centering windows on Poke-Out (rather than, or in
addition to, Poke-In) changes decoding accuracy, before any universal-window redesign is finalized. If
the second event turns out NOT to be a clean Poke-Out, that's equally important to know before building
anything around it, and the existing Poke-In-based approach stays as the trusted reference point.
